# TP Jour 3 — Exploration et transformation des sources TPCx-BB

**Bilal El Hantlaoui** — 18/11/2026
Mastère Développement, Data & IA — Retail Intelligence Platform — UC-06

## Objectif

Lire les quatre sources retenues depuis la zone Bronze HDFS posée au Jour 2,
contrôler leur schéma, leur volume et leurs anomalies, vérifier les relations
entre elles, puis produire un indicateur mensuel par magasin et l'enregistrer
en Parquet.

Les quatre sources : `store_sales` (faits), `date_dim`, `store` et `item`
(dimensions). Les gros fichiers du kit (`inventory`, `web_clickstreams`) sont
écartés : volume inutile aux questions traitées, et aucun identifiant fiable
ne relie la navigation web aux ventes.

**Règle tenue dans tout le notebook** : aucune ligne n'est supprimée, aucune
valeur n'est remplacée sans règle écrite, et aucune jointure n'est faite sur
une ressemblance de noms.

## 1. Session Spark

Deux réglages ne sont pas évidents et méritent une explication.

`HADOOP_USER_NAME=root` : les répertoires `/retail` de HDFS ont été créés par
le conteneur Hadoop, qui tourne en root. Un client qui se présente sous un
autre nom n'a pas le droit d'y écrire.

`dfs.client.use.datanode.hostname=true` : le NameNode répond bien sur
`localhost:9000`, mais il communique au client l'adresse interne du DataNode
dans le réseau Docker, que macOS ne sait pas joindre. Ce réglage demande au
client d'utiliser le nom d'hôte publié à la place.

Le mode est `local[*]` : le calcul se fait sur cette machine, avec ses
processeurs. Le stockage, lui, reste distribué puisque les données sont dans
HDFS. Aucun cluster Spark n'est utilisé.

In [2]:
import os
os.environ["HADOOP_USER_NAME"] = "root"

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("TP3-Exploration")
         .master("local[*]")
         .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

BRONZE = "hdfs://localhost:9000/retail/bronze/data"
GOLD = "hdfs://localhost:9000/retail/gold"

print("Spark", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 15:05:25 WARN Utils: Your hostname, MacBook-Pro-de-le.local, resolves to a loopback address: 127.0.0.1; using 10.74.1.185 instead (on interface en0)
26/11/18 15:05:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 15:05:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.3.0


## 2. Lecture brute de la table de faits

On lit **sans imposer de noms de colonnes**. C'est volontaire : les fichiers
`.dat` de TPCx-BB n'ont pas de ligne d'en-tête, donc les noms ne sont nulle
part dans les données. On regarde d'abord ce qu'on a reçu.

`inferSchema=True` demande à Spark de proposer des types. C'est une
proposition utile pour explorer, pas une validation : les types définitifs
seront déclarés explicitement avant d'écrire la zone Silver.

In [3]:
CHEMIN_VENTES = f"{BRONZE}/store_sales/store_sales.dat"

brut = (spark.read
        .option("header", False)
        .option("sep", "|")
        .option("inferSchema", True)
        .csv(CHEMIN_VENTES))

print("colonnes lues :", len(brut.columns))
print("lignes        :", brut.count())
brut.show(3, truncate=False)

colonnes lues : 23
lignes        : 667579
+-----+-----+-----+-----+------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|_c0  |_c1  |_c2  |_c3  |_c4   |_c5 |_c6  |_c7|_c8|_c9|_c10|_c11 |_c12  |_c13 |_c14   |_c15   |_c16   |_c17   |_c18  |_c19  |_c20   |_c21   |_c22   |
+-----+-----+-----+-----+------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|36890|71480|10001|44404|493944|2898|44093|10 |294|0  |31  |18.0 |30.78 |23.08|238.7  |715.48 |558.0  |954.18 |21.46 |0.0   |715.48 |736.94 |157.48 |
|36890|71480|10000|44404|493944|2898|44093|10 |279|0  |54  |99.21|148.82|83.34|3535.92|4500.36|5357.34|8036.28|225.02|0.0   |4500.36|4725.38|-856.98|
|36890|71480|15556|44404|493944|2898|44093|10 |114|0  |39  |79.49|94.59 |83.24|442.65 |3246.36|3100.11|3689.01|184.07|616.81|2629.55|2813.62|-470.56|
+-----+-----+-----+-----+------+----+-----+---+---+---+---

### Contrôle du séparateur de fin de ligne

Les fichiers `.dat` de certains kits terminent chaque ligne par le séparateur,
ce qui fait apparaître une colonne vide de plus que le schéma. Ici le compte
est de 23 colonnes pour 23 attendues : **ce kit n'a pas de séparateur final**.
À noter dans les anomalies, avec la mention « néant ».

## 3. Application des noms de colonnes, puis preuve

Le nombre de colonnes correspond à la spécification, on peut donc appliquer
les noms officiels. Mais appliquer n'est pas prouver : le TP interdit de
construire sur une supposition.

La preuve est arithmétique. Sur une ligne de vente, le montant étendu vaut la
quantité multipliée par le prix unitaire. Si l'identité tient sur un
échantillon de lignes réelles, alors ces trois colonnes sont bien celles que
je crois, et une colonne mal placée ferait échouer le test.

In [4]:
noms = ["ss_sold_date_sk", "ss_sold_time_sk", "ss_item_sk", "ss_customer_sk",
        "ss_cdemo_sk", "ss_hdemo_sk", "ss_addr_sk", "ss_store_sk",
        "ss_promo_sk", "ss_ticket_number", "ss_quantity", "ss_wholesale_cost",
        "ss_list_price", "ss_sales_price", "ss_ext_discount_amt",
        "ss_ext_sales_price", "ss_ext_wholesale_cost", "ss_ext_list_price",
        "ss_ext_tax", "ss_coupon_amt", "ss_net_paid", "ss_net_paid_inc_tax",
        "ss_net_profit"]

assert len(noms) == len(brut.columns), "compte de colonnes non conforme"
ventes = brut.toDF(*noms).cache()

ventes.printSchema()

root
 |-- ss_sold_date_sk: integer (nullable = true)
 |-- ss_sold_time_sk: integer (nullable = true)
 |-- ss_item_sk: integer (nullable = true)
 |-- ss_customer_sk: integer (nullable = true)
 |-- ss_cdemo_sk: integer (nullable = true)
 |-- ss_hdemo_sk: integer (nullable = true)
 |-- ss_addr_sk: integer (nullable = true)
 |-- ss_store_sk: integer (nullable = true)
 |-- ss_promo_sk: integer (nullable = true)
 |-- ss_ticket_number: integer (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: double (nullable = true)
 |-- ss_list_price: double (nullable = true)
 |-- ss_sales_price: double (nullable = true)
 |-- ss_ext_discount_amt: double (nullable = true)
 |-- ss_ext_sales_price: double (nullable = true)
 |-- ss_ext_wholesale_cost: double (nullable = true)
 |-- ss_ext_list_price: double (nullable = true)
 |-- ss_ext_tax: double (nullable = true)
 |-- ss_coupon_amt: double (nullable = true)
 |-- ss_net_paid: double (nullable = true)
 |-- ss_net_paid_inc_tax

In [5]:
echantillon = ventes.limit(1000)
conformes = echantillon.filter(
    F.abs(F.col("ss_ext_sales_price")
          - F.col("ss_quantity") * F.col("ss_sales_price")) < 0.02).count()

print("ss_ext_sales_price = ss_quantity x ss_sales_price")
print("verifie sur", conformes, "/", echantillon.count(), "lignes testees")

ss_ext_sales_price = ss_quantity x ss_sales_price
verifie sur 1000 / 1000 lignes testees


## 4. Valeurs manquantes

Une valeur nulle n'est pas forcément une erreur : elle peut signifier une
information inconnue, non applicable ou légitimement absente. On compte
d'abord, on décide ensuite.

Le comptage se fait **en une seule passe** : une agrégation qui additionne les
nuls de toutes les colonnes à la fois. Une boucle avec un `count()` par
colonne donnerait le même résultat mais relirait le fichier 23 fois.

In [6]:
nulls = ventes.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ventes.columns
]).collect()[0].asDict()

nb = ventes.count()
for colonne, compte in nulls.items():
    if compte:
        print(f"{colonne:<24}{compte:>8}  ({100 * compte / nb:.2f} %)")

ss_cdemo_sk                  238  (0.04 %)
ss_hdemo_sk                  295  (0.04 %)
ss_addr_sk                   448  (0.07 %)
ss_promo_sk                  158  (0.02 %)


**Lecture du résultat.** Les colonnes concernées sont des clés secondaires :
démographie du client, démographie du foyer, adresse, promotion. Moins de
0,1 % des ventes dans chaque cas.

**Décision : conserver.** Une vente sans profil démographique et sans
promotion reste une vente. Remplacer ces nuls par zéro créerait un faux
client et une fausse promotion, ce qui fausserait toute analyse ultérieure.

## 5. Doublons et clé candidate

La spécification TPC-DS déclare `(ss_item_sk, ss_ticket_number)` comme clé
primaire. Avant d'écrire la moindre règle, je teste.

Trois questions dans l'ordre : la clé déclarée est-elle unique ? des lignes
sont-elles répétées à l'identique ? et si la clé n'est pas unique, qu'est-ce
qui distingue les lignes concernées ?

In [7]:
for cle in (["ss_ticket_number"],
            ["ss_ticket_number", "ss_item_sk"],
            ["ss_sold_date_sk", "ss_ticket_number", "ss_item_sk"],
            ["ss_sold_date_sk", "ss_store_sk", "ss_ticket_number", "ss_item_sk"]):
    distinctes = ventes.select(cle).distinct().count()
    print(f"{' + '.join(cle):<58}{distinctes:>8}  ecart {nb - distinctes}")

print()
print("lignes distinctes sur les 23 colonnes :", ventes.distinct().count())
print("lignes repetees a l'identique         :", nb - ventes.distinct().count())

ss_ticket_number                                             89100  ecart 578479
ss_ticket_number + ss_item_sk                               663443  ecart 4136
ss_sold_date_sk + ss_ticket_number + ss_item_sk             663443  ecart 4136
ss_sold_date_sk + ss_store_sk + ss_ticket_number + ss_item_sk  663443  ecart 4136

lignes distinctes sur les 23 colonnes : 667579
lignes repetees a l'identique         : 0


In [8]:
groupe = (ventes.groupBy("ss_ticket_number", "ss_item_sk")
          .count().filter("count > 1").orderBy(F.desc("count")).first())

extrait = ventes.filter((F.col("ss_ticket_number") == groupe[0])
                        & (F.col("ss_item_sk") == groupe[1])).collect()

print(f"ticket {groupe[0]}, article {groupe[1]} : {groupe[2]} lignes\n")
for colonne in noms:
    valeurs = [str(ligne[colonne]) for ligne in extrait]
    marque = "   <-- DIFFERENT" if len(set(valeurs)) > 1 else ""
    print(f"{colonne:<24}{' | '.join(valeurs)}{marque}")

ticket 2378, article 10003 : 4 lignes

ss_sold_date_sk         36938 | 36938 | 36938 | 36938
ss_sold_time_sk         29366 | 29366 | 29366 | 29366
ss_item_sk              10003 | 10003 | 10003 | 10003
ss_customer_sk          53484 | 53484 | 53484 | 53484
ss_cdemo_sk             1762419 | 1762419 | 1762419 | 1762419
ss_hdemo_sk             7076 | 7076 | 7076 | 7076
ss_addr_sk              21298 | 21298 | 21298 | 21298
ss_store_sk             6 | 6 | 6 | 6
ss_promo_sk             169 | 254 | 231 | 296   <-- DIFFERENT
ss_ticket_number        2378 | 2378 | 2378 | 2378
ss_quantity             9 | 64 | 63 | 61   <-- DIFFERENT
ss_wholesale_cost       77.97 | 3.61 | 30.41 | 36.82   <-- DIFFERENT
ss_list_price           99.8 | 6.32 | 31.63 | 60.38   <-- DIFFERENT
ss_sales_price          89.82 | 2.02 | 27.2 | 35.02   <-- DIFFERENT
ss_ext_discount_amt     89.82 | 275.2 | 279.09 | 1546.96   <-- DIFFERENT
ss_ext_sales_price      808.38 | 129.28 | 1713.6 | 2136.22   <-- DIFFERENT
ss_ext_wholesale_co

**Lecture du résultat.** Aucune ligne n'est répétée à l'identique, et ajouter
la date et le magasin à la clé ne réduit pas l'écart : les lignes concernées
partagent donc le même ticket, le même article, la même date et le même
magasin. Ce qui diffère, ce sont la quantité, le prix de vente, la promotion
et les montants.

**Conclusion.** C'est le même article vendu plusieurs fois sur un même ticket,
en lignes séparées, avec une promotion et un prix différents. Cas réel en
caisse. Le grain de la table est donc **la ligne de vente**, plus fin que le
couple (ticket, article), et cette table n'a pas de clé naturelle unique dans
les données fournies.

**Décision : aucune déduplication.** L'écart est conservé comme anomalie
documentée. Supprimer ces lignes supprimerait des ventes réelles.

## 6. Les dimensions et leurs clés

Une dimension dont la clé n'est pas unique multiplie les lignes de faits au
moment de la jointure, et fausse tous les montants sans aucun message
d'erreur. Donc on vérifie avant, pas après.

In [9]:
def charger(table, colonnes):
    return (spark.read.option("header", False).option("sep", "|")
            .option("inferSchema", True).csv(f"{BRONZE}/{table}/{table}.dat")
            .toDF(*colonnes))

noms_item = ["i_item_sk", "i_item_id", "i_rec_start_date", "i_rec_end_date",
             "i_item_desc", "i_current_price", "i_wholesale_cost",
             "i_brand_id", "i_brand", "i_class_id", "i_class", "i_category_id",
             "i_category", "i_manufact_id", "i_manufact", "i_size",
             "i_formulation", "i_color", "i_units", "i_container",
             "i_manager_id", "i_product_name"]

noms_store = ["s_store_sk", "s_store_id", "s_rec_start_date", "s_rec_end_date",
              "s_closed_date_sk", "s_store_name", "s_number_employees",
              "s_floor_space", "s_hours", "s_manager", "s_market_id",
              "s_geography_class", "s_market_desc", "s_market_manager",
              "s_division_id", "s_division_name", "s_company_id",
              "s_company_name", "s_street_number", "s_street_name",
              "s_street_type", "s_suite_number", "s_city", "s_county",
              "s_state", "s_zip", "s_country", "s_gmt_offset",
              "s_tax_precentage"]

noms_date = ["d_date_sk", "d_date_id", "d_date", "d_month_seq", "d_week_seq",
             "d_quarter_seq", "d_year", "d_dow", "d_moy", "d_dom", "d_qoy",
             "d_fy_year", "d_fy_quarter_seq", "d_fy_week_seq", "d_day_name",
             "d_quarter_name", "d_holiday", "d_weekend", "d_following_holiday",
             "d_first_dom", "d_last_dom", "d_same_day_ly", "d_same_day_lq",
             "d_current_day", "d_current_week", "d_current_month",
             "d_current_quarter", "d_current_year"]

item = charger("item", noms_item)
magasin = charger("store", noms_store)
calendrier = charger("date_dim", noms_date)

for nom, df, cle in (("item", item, "i_item_sk"),
                     ("store", magasin, "s_store_sk"),
                     ("date_dim", calendrier, "d_date_sk")):
    lignes = df.count()
    distinctes = df.select(cle).distinct().count()
    verdict = "cle unique" if lignes == distinctes else "CLE NON UNIQUE"
    print(f"{nom:<12}{lignes:>8} lignes  {distinctes:>8} cles distinctes  -> {verdict}")

item           17820 lignes     17820 cles distinctes  -> cle unique
store             12 lignes        12 cles distinctes  -> cle unique
date_dim      109573 lignes    109573 cles distinctes  -> cle unique


## 7. Intégrité référentielle

Combien de ventes pointent vers un produit, un magasin ou une date qui
n'existe pas au référentiel ? `left_anti` garde les lignes de gauche **sans**
correspondance à droite : c'est le contrôle d'intégrité en une ligne.

Ce chiffre décide du type de jointure à utiliser ensuite. S'il est nul, une
jointure `inner` ne perd rien.

In [10]:
for nom, df, cle_dim, cle_vente in (
        ("item", item, "i_item_sk", "ss_item_sk"),
        ("store", magasin, "s_store_sk", "ss_store_sk"),
        ("date_dim", calendrier, "d_date_sk", "ss_sold_date_sk")):
    orphelins = ventes.join(df.select(cle_dim),
                            ventes[cle_vente] == df[cle_dim],
                            "left_anti").count()
    nuls = ventes.filter(F.col(cle_vente).isNull()).count()
    print(f"{cle_vente:<20}-> {nom:<10}sans correspondance : {orphelins:<8}"
          f"cle nulle : {nuls}")

ss_item_sk          -> item      sans correspondance : 0       cle nulle : 0
ss_store_sk         -> store     sans correspondance : 0       cle nulle : 0
ss_sold_date_sk     -> date_dim  sans correspondance : 0       cle nulle : 0


## 8. Nettoyage

Deux règles seulement sur cette table, et chacune se justifie.

**Harmonisation des libellés** : les espaces de bord sont retirés des colonnes
texte. C'est une mise en forme, aucune information n'est perdue.

**Montants négatifs** : ils sont **conservés** et marqués par une colonne
dédiée. Un profit net négatif est une vente à perte, cas métier réel. Le
supprimer effacerait de l'information et biaiserait la marge. On documente, on
ne purge pas.

In [11]:
colonnes_texte = [c.name for c in ventes.schema.fields
                  if c.dataType.simpleString() == "string"]

propres = ventes
for colonne in colonnes_texte:
    propres = propres.withColumn(colonne, F.trim(F.col(colonne)))

propres = propres.withColumn(
    "ind_montant_negatif",
    (F.col("ss_net_paid") < 0) | (F.col("ss_net_profit") < 0))

print("colonnes texte harmonisees :", len(colonnes_texte))
print("lignes a montant negatif   :", propres.filter("ind_montant_negatif").count())
print("lignes conservees          :", propres.count(), "sur", nb)

for colonne in ("ss_quantity", "ss_net_paid", "ss_net_profit"):
    stats = propres.agg(F.min(colonne), F.max(colonne)).collect()[0]
    print(f"{colonne:<18}min {stats[0]:>12}   max {stats[1]:>12}")

colonnes texte harmonisees : 0
lignes a montant negatif   : 495971
lignes conservees          : 667579 sur 667579
ss_quantity       min            1   max          100
ss_net_paid       min          0.0   max      18745.0
ss_net_profit     min      -9866.6   max       9134.0


## 9. Jointure vérifiée

Les relations sont maintenant démontrées : clés de dimension uniques, aucun
orphelin. Je peux joindre.

`date_dim` en `inner` : une vente sans date valide ne pourrait pas être placée
dans un mois. Le contrôle précédent montre qu'il n'y en a aucune, donc cette
jointure ne perd rien.

`store` en `left` : une vente dont le magasin serait absent du référentiel
resterait du chiffre d'affaires réel. En `inner`, je la perdrais en silence.

Le contrôle qui compte : **le nombre de lignes avant et après doit être
identique**. S'il augmente, une clé de dimension n'est pas unique.

In [12]:
enrichi = (propres
           .join(calendrier.select("d_date_sk", "d_date", "d_year", "d_moy"),
                 propres.ss_sold_date_sk == F.col("d_date_sk"), "inner")
           .join(magasin.select("s_store_sk", "s_store_name", "s_state"),
                 propres.ss_store_sk == F.col("s_store_sk"), "left"))

print("lignes avant jointures :", propres.count())
print("lignes apres jointures :", enrichi.count())

bornes = enrichi.agg(F.min("d_date"), F.max("d_date")).collect()[0]
print("periode couverte       :", bornes[0], "->", bornes[1])
print("magasins presents      :", enrichi.select("s_store_sk").distinct().count())

lignes avant jointures : 667579
lignes apres jointures : 667579
periode couverte       : 2001-01-01 -> 2005-12-13
magasins presents      : 12


## 10. Agrégation : CA mensuel par magasin

Cet indicateur répond à deux questions du Jour 1, toutes deux classées
indispensables : Q1 sur l'évolution du chiffre d'affaires, Q2 sur la
contribution de chaque magasin.

**Grain du résultat : une ligne = un mois × un magasin.**

Trois précautions. Le nombre de transactions se compte en `countDistinct` sur
le numéro de ticket, et non en `count(*)` : une ligne de vente n'est pas une
transaction. Le panier moyen est le CA divisé par ce nombre de transactions,
exactement la formule du KPI 3 du dictionnaire du Jour 1. Et `try_divide`
renvoie une valeur nulle au lieu de lever une erreur si un groupe n'avait
aucune transaction.

In [13]:
mensuel = (enrichi
           .groupBy("d_year", "d_moy", "s_store_sk", "s_store_name")
           .agg(F.round(F.sum("ss_net_paid"), 2).alias("ca_net"),
                F.sum("ss_quantity").alias("quantite"),
                F.countDistinct("ss_ticket_number").alias("nb_transactions"),
                F.count("*").alias("nb_lignes_vente"))
           .withColumn("panier_moyen",
                       F.round(F.col("ca_net") / F.col("nb_transactions"), 2)))

fenetre = Window.partitionBy("s_store_sk").orderBy("d_year", "d_moy")
mensuel = (mensuel
           .withColumn("ca_mois_precedent", F.lag("ca_net").over(fenetre))
           .withColumn("evolution_pct",
                       F.round(100 * (F.col("ca_net") - F.col("ca_mois_precedent"))
                               / F.col("ca_mois_precedent"), 1)))

print("lignes du resultat :", mensuel.count(), "(grain : un mois x un magasin)")
mensuel.orderBy("d_year", "d_moy", F.desc("ca_net")).show(10, truncate=False)

lignes du resultat : 720 (grain : un mois x un magasin)
+------+-----+----------+------------+----------+--------+---------------+---------------+------------+-----------------+-------------+
|d_year|d_moy|s_store_sk|s_store_name|ca_net    |quantite|nb_transactions|nb_lignes_vente|panier_moyen|ca_mois_precedent|evolution_pct|
+------+-----+----------+------------+----------+--------+---------------+---------------+------------+-----------------+-------------+
|2001  |1    |9         |Germantown  |1792112.77|53660   |142            |1067           |12620.51    |NULL             |NULL         |
|2001  |1    |8         |Williamstown|1733989.6 |52840   |142            |1059           |12211.19    |NULL             |NULL         |
|2001  |1    |11        |Bedford     |1696543.33|48894   |129            |974            |13151.5     |NULL             |NULL         |
|2001  |1    |5         |Egypt       |1671777.21|50160   |134            |1033           |12475.95    |NULL             |NULL   

### Contrôle de bouclage

Une agrégation ne doit ni perdre ni dupliquer. La somme des CA mensuels doit
donc égaler la somme des montants du détail, au centime. Si ce contrôle passe,
ni la jointure ni le regroupement n'ont altéré les données.

In [14]:
total_detail = propres.agg(F.round(F.sum("ss_net_paid"), 2)).collect()[0][0]
total_agrege = mensuel.agg(F.round(F.sum("ca_net"), 2)).collect()[0][0]

print("somme du detail  :", total_detail)
print("somme des agrege :", total_agrege)
print("ecart            :", round(abs(total_detail - total_agrege), 2))

somme du detail  : 1150529586.37
somme des agrege : 1150529586.37
ecart            : 0.0


## 11. Écriture en Parquet, puis relecture

Parquet est un format de stockage **colonnaire** : une requête qui ne lit que
deux colonnes sur dix ne lit effectivement que ces deux colonnes. Il
transporte aussi son schéma, donc la relecture n'a besoin d'aucune inférence
de type.

L'écriture ne suffit pas : le TP demande de **relire et vérifier**. Trois
contrôles après écriture — la lecture fonctionne, le schéma est conservé, le
nombre de lignes est cohérent.

`mode("overwrite")` remplace le contenu de la destination. Il n'est utilisé
que sur un répertoire de sortie dédié, jamais sur les données sources.

In [15]:
DESTINATION = f"{GOLD}/ca_mensuel_magasin_notebook"

mensuel.write.mode("overwrite").parquet(DESTINATION)

relu = spark.read.parquet(DESTINATION)

ecrits = [(c.name, c.dataType) for c in mensuel.schema.fields]
relus = [(c.name, c.dataType) for c in relu.schema.fields]

print("relecture                    : OK")
print("lignes relues                :", relu.count(), "/ ecrites", mensuel.count())
print("colonnes et types conserves  :", ecrits == relus)
print("(la nullabilite peut differer : Parquet la redefinit a la relecture)")
relu.printSchema()

relecture                    : OK
lignes relues                : 720 / ecrites 720
colonnes et types conserves  : True
(la nullabilite peut differer : Parquet la redefinit a la relecture)
root
 |-- d_year: integer (nullable = true)
 |-- d_moy: integer (nullable = true)
 |-- s_store_sk: integer (nullable = true)
 |-- s_store_name: string (nullable = true)
 |-- ca_net: double (nullable = true)
 |-- quantite: long (nullable = true)
 |-- nb_transactions: long (nullable = true)
 |-- nb_lignes_vente: long (nullable = true)
 |-- panier_moyen: double (nullable = true)
 |-- ca_mois_precedent: double (nullable = true)
 |-- evolution_pct: double (nullable = true)



## 12. Conclusion

### Ce que j'ai constaté

| Mesure | Valeur |
| --- | --- |
| Lignes de vente | 667 579 |
| Tickets distincts | 89 100, soit 7,5 articles par ticket |
| Période couverte | 2001 à 2005 |
| Magasins | 12 sur 12 présents dans les ventes |
| CA net total | 1 150 529 586,37 |
| Orphelins vers les dimensions | 0 sur les trois relations |
| Valeurs nulles | clés secondaires uniquement, moins de 0,1 % |
| Profit net négatif | 495 971 lignes, soit 74,3 % |

### Les trois décisions que j'ai prises, et pourquoi

**Les noms de colonnes sont prouvés, pas supposés.** Le compte de colonnes
correspond à la spécification et l'identité quantité × prix est vérifiée sur
1000 lignes sur 1000. Sans cette preuve, tout le reste reposerait sur une
hypothèse.

**Aucune ligne n'est supprimée.** Les 4 136 lignes partageant un couple
(ticket, article) ne sont pas des doublons : aucune ligne n'est répétée à
l'identique et elles diffèrent sur les prix et les quantités. Le grain réel
est la ligne de vente. Les montants négatifs sont conservés et marqués : une
vente à perte est un fait, pas une erreur.

**Chaque transformation est bouclée.** Le nombre de lignes est identique avant
et après jointure, et la somme des agrégats égale la somme du détail à zéro
près. Ce sont ces deux égalités qui garantissent que l'indicateur est juste.

### Limites de ce travail

Jeu de données **synthétique** : aucune conclusion commerciale ne peut en être
tirée, et le profit net négatif sur 74 % des lignes en est l'illustration.
Périmètre limité aux ventes magasin : le canal web, les retours et les stocks
ne sont pas traités. L'unité monétaire n'est pas documentée dans le kit. Le
calcul s'exécute en local, pas sur un cluster Spark, même si le stockage reste
distribué via HDFS. Enfin, Parquet n'apporte aucune garantie
transactionnelle : ni versions, ni retour arrière, ni écriture atomique — cela
demanderait une couche format de table comme Delta Lake ou Iceberg.

In [16]:
spark.stop()
print("session fermee")

session fermee
